## Tools

Tools in LLMs are external functions, APIs, or systems that the model can use to perform actions beyond text generation.

LLMs by themselves only predict text.
Tools give them capabilities like:

    Searching the web
    Running code
    Querying databases
    Calling APIs
    Using calculators
    Accessing files

In [3]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")


llm=ChatGroq(model="llama-3.1-8b-instant")

In [4]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather of a location"""
    return f"It's sunny in {location}"

In [5]:
model_with_tools=llm.bind_tools([get_weather])


In [6]:
response=model_with_tools.invoke("What is the weather in Mumbai")

print(response)

content='' additional_kwargs={'tool_calls': [{'id': '5x6bgb5nz', 'function': {'arguments': '{"location":"Mumbai"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 218, 'total_tokens': 233, 'completion_time': 0.027830425, 'completion_tokens_details': None, 'prompt_time': 0.014715933, 'prompt_tokens_details': None, 'queue_time': 0.158553058, 'total_time': 0.042546358}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e6d57-8a67-7a21-be5c-fd2d2e15100d-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Mumbai'}, 'id': '5x6bgb5nz', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 218, 'output_tokens': 15, 'total_tokens': 233}


In [7]:
for tool_call in response.tool_calls:
    print(f"Tool:{tool_call['name']}")
    print(f"Args:{tool_call['args']}")


Tool:get_weather
Args:{'location': 'Mumbai'}


### Tool Execution loop

In [17]:
## Model generates tool calls
messages=[{"role":"user","content":"What is the weather in Boston"}]
ai_msg=model_with_tools.invoke(messages)
messages.append(ai_msg)

## Check whether the ai_msg contains the tool_calls
for tool_call in ai_msg.tool_calls:
    ## Execute the tool with the generated arguments
    tool_result=get_weather.invoke(tool_call)
    messages.append(tool_result)

## Pass the messages back to the model to get the final response
final_response=model_with_tools.invoke(messages)
print(final_response.content)

However, please note that the function 'get_weather' was not defined in the code snippet provided. The response 'It's sunny in Boston' is just an example result.


In [18]:
messages

[{'role': 'user', 'content': 'What is the weather in Boston'},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '1z59d7jfp', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 218, 'total_tokens': 232, 'completion_time': 0.027041426, 'completion_tokens_details': None, 'prompt_time': 0.013785896, 'prompt_tokens_details': None, 'queue_time': 0.158499277, 'total_time': 0.040827322}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e6e00-a323-7b02-9498-ebf365475ea3-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '1z59d7jfp', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 218, 'output_tokens': 14, 'total_tokens': 232}),
 ToolMessage(content="It's su